# 06. 영상 수집 (세션 A)
- statcast parquet → play_id 추출 → CDN URL 크롤링 → MP4 다운로드 → Drive zip 저장
- **세션 A** 만 이 노트북 실행. 세션 B(스켈레톤), 세션 C(YOLO)는 zip을 읽어서 처리

### 슬롯 배분
| 슬롯 | 담당 시즌 |
|------|-----------|
| 0 | 2021 |
| 1 | 2022 |
| 2 | 2023 |
| 3 | 2024 |
| 4 | 2025 |

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# ══════════════════════════════════════════════════════
# ★ 여기만 수정 ★
MY_SLOT     = 0    # 0~4 (총 5개 세션으로 분할)
TOTAL_SLOTS = 5
BATCH_SIZE  = 200  # zip 한 개에 담을 MP4 수

# play_ids_all.csv Drive 파일 ID (기존 01_video_download.ipynb 에서 쓰던 것)
PLAY_ID_ALL_FILE_ID = '1hXM8ixbPg14c5ih4AWpjGheewELRCG8M'
# ══════════════════════════════════════════════════════

import os

BASE         = '/content/drive/MyDrive/MLB_pitcher'
DATA_DIR     = os.path.join(BASE, 'data')
VIDEO_DIR    = os.path.join(BASE, 'video_zips')
TEMP_DIR     = '/content/tmp_videos'

PLAY_ID_ALL  = '/content/play_ids_all.csv'
PROGRESS_CSV = os.path.join(DATA_DIR, f'progress_slot_{MY_SLOT}.csv')

for d in [DATA_DIR, VIDEO_DIR, TEMP_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'슬롯 {MY_SLOT} / 총 {TOTAL_SLOTS}개')
print(f'progress : {PROGRESS_CSV}')
print(f'zip 저장 : {VIDEO_DIR}')

슬롯 0 / 총 5개
progress : /content/drive/MyDrive/MLB_pitcher/data/progress_slot_0.csv
zip 저장 : /content/drive/MyDrive/MLB_pitcher/video_zips


In [3]:
!pip install -q requests beautifulsoup4 gdown pandas pyarrow

import re
import time
import zipfile
import shutil
import requests
import pandas as pd
import gdown
from bs4 import BeautifulSoup
from pathlib import Path

REQ_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}
print('라이브러리 로드 완료')

라이브러리 로드 완료


## Step 1. play_ids_all.csv 로드 (Drive에서 다운로드)

In [4]:
import requests
import re
import time
import shutil
import pandas as pd
import gdown
from bs4 import BeautifulSoup

# ── 설정 ──────────────────────────────────────────────
GAMES_PER_PITCHER = 5   # 선수당 연도별 최대 경기 수
PITCHES_PER_GAME  = 15  # 경기당 초구(1st pitch) 수

PARQUET_FILE_IDS = {
    2021: '1KT17izBcO92VXTIr0LyyWGl-0bKHW7sS',
    2022: '16ak1N09M0ptyq4xjR-mpCBnY-L1dQadE',
    2023: '1bX2n3hpptTBZszPyB5EY2Ov0lCI4nM8I',
    2024: '17-ZhSO18-vjkSFXOZtQKEVTYnPmGjMuY',
    2025: '1uW_apJ4IC63FvTvK6BkZYBaAKA8ge5_L',
}
PARQUET_DIR = '/content/parquets'
os.makedirs(PARQUET_DIR, exist_ok=True)

SAMPLE_CSV = os.path.join(DATA_DIR, 'play_ids_sample.csv')
CRAWL_CKPT = os.path.join(DATA_DIR, 'crawl_ckpt_sample.csv')

REQ_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def crawl_play_ids_for_game(pitcher_id, game_pk, season, retry=3):
    """투수 + 경기 기준으로 해당 경기 초구 play_id 수집"""
    url = (
        f'https://baseballsavant.mlb.com/statcast_search'
        f'?hfPT=&hfAB=&hfGT=R%7C&hfPR=&hfZ=&hfStadium=&hfBBL=&hfNewZones='
        f'&hfPull=&hfC=&hfSea={season}%7C&hfSit=&player_type=pitcher'
        f'&hfOuts=&home_road=&pitcher_throws=&batter_stands=&hfSA='
        f'&hfEventOuts=&hfEventRuns=&game_date_gt=&game_date_lt=&hfMo='
        f'&hfTeam=&hfOpponent=&hfRO=&position=&hfInfield=&hfOutfield='
        f'&hfInn=&hfBBT=&hfFlag='
        f'&pitchers_lookup%5B%5D={pitcher_id}'
        f'&game_pk={game_pk}'
        f'&metric_1=&group_by=name&min_pitches=0&min_results=0&min_pas=0'
        f'&sort_col=pitches&player_event_sort=api_p_release_speed&sort_order=desc'
        f'&type=details&player_id={pitcher_id}'
    )
    for attempt in range(retry):
        try:
            res = requests.get(url, headers=REQ_HEADERS, timeout=30)
            if res.status_code != 200:
                time.sleep(2 ** attempt)
                continue
            soup = BeautifulSoup(res.text, 'html.parser')
            ids = []
            for tag in soup.find_all('a', href=True):
                m = re.search(r'playId=([0-9a-f\-]{36})', tag['href'])
                if m:
                    ids.append(m.group(1))
            return list(set(ids))
        except Exception:
            time.sleep(2 ** attempt)
    return []

# ── SAMPLE_CSV 이미 있으면 로드 ────────────────────────
if os.path.exists(SAMPLE_CSV):
    play_df = pd.read_csv(SAMPLE_CSV)
    print(f'기존 샘플 로드: {len(play_df):,}개')

else:
    # ── Step 1. parquet → 선발투수 × 5경기 × 초구 목록 ──
    target_records = []

    for season, file_id in PARQUET_FILE_IDS.items():
        parquet_path = os.path.join(PARQUET_DIR, f'statcast_{season}.parquet')
        if not os.path.exists(parquet_path):
            print(f'[{season}] parquet 다운로드 중...')
            gdown.download(f'https://drive.google.com/uc?id={file_id}', parquet_path, quiet=False)

        print(f'[{season}] 필터링 중...')
        df = pd.read_parquet(parquet_path, columns=[
            'game_pk', 'game_date', 'game_type',
            'pitcher', 'pitch_number', 'at_bat_number',
        ])

        # 정규시즌 + 초구만
        df = df[(df['game_type'] == 'R') & (df['pitch_number'] == 1)]

        # 선발투수 추출
        ab_min = (
            df.groupby(['game_pk', 'pitcher'])['at_bat_number']
            .min().reset_index()
            .rename(columns={'at_bat_number': 'min_ab'})
        )
        starter_games = ab_min[ab_min['min_ab'] == 1][['game_pk', 'pitcher']]
        df = df.merge(starter_games, on=['game_pk', 'pitcher'], how='inner')

        # 선수당 최대 5경기 샘플링
        sampled_games = (
            df.groupby('pitcher')['game_pk']
            .apply(lambda x: x.drop_duplicates().sample(
                min(GAMES_PER_PITCHER, x.nunique()), random_state=42
            ))
            .reset_index(level=0, drop=True)
        )
        df = df[df['game_pk'].isin(sampled_games)]

        for (pitcher, game_pk), _ in df.groupby(['pitcher', 'game_pk']):
            target_records.append({
                'pitcher_id': pitcher,
                'game_pk'   : game_pk,
                'season'    : season,
            })

        print(f'[{season}] 투수×경기 조합: {df.groupby(["pitcher","game_pk"]).ngroups:,}건')

    target_df = pd.DataFrame(target_records).drop_duplicates().reset_index(drop=True)
    print(f'\n전체 크롤링 대상: {len(target_df):,}건')
    print(f'예상 시간: 약 {len(target_df) / 3600:.1f}시간')

    # ── Step 2. 체크포인트 로드 ───────────────────────────
    if os.path.exists(CRAWL_CKPT):
        done_df   = pd.read_csv(CRAWL_CKPT)
        done_keys = set(zip(done_df['pitcher_id'], done_df['game_pk']))
        play_recs = done_df.to_dict('records')
        print(f'체크포인트 로드: {len(done_keys):,}건 완료')
    else:
        done_keys = set()
        play_recs = []

    remaining = target_df[
        ~target_df.apply(lambda r: (r['pitcher_id'], r['game_pk']) in done_keys, axis=1)
    ].reset_index(drop=True)
    print(f'남은 크롤링: {len(remaining):,}건')

    # ── Step 3. 크롤링 ────────────────────────────────────
    for i, row in remaining.iterrows():
        ids = crawl_play_ids_for_game(row['pitcher_id'], row['game_pk'], row['season'])
        # 경기당 최대 15개
        for play_id in ids[:PITCHES_PER_GAME]:
            play_recs.append({
                'play_id'   : play_id,
                'pitcher_id': row['pitcher_id'],
                'game_pk'   : row['game_pk'],
                'season'    : row['season'],
            })
        done_keys.add((row['pitcher_id'], row['game_pk']))

        if (i + 1) % 50 == 0 or (i + 1) == len(remaining):
            pd.DataFrame(play_recs).to_csv(CRAWL_CKPT, index=False)
            print(f'  [{i+1}/{len(remaining)}] {len(play_recs):,}개 play_id 누적')

        time.sleep(1.0)

    play_df = (
        pd.DataFrame(play_recs)
        .drop_duplicates('play_id')
        .reset_index(drop=True)
    )
    play_df.to_csv(SAMPLE_CSV, index=False)
    print(f'\n샘플 저장 완료: {len(play_df):,}개 → {SAMPLE_CSV}')

# ── 슬롯 분할 ─────────────────────────────────────────
TOTAL     = len(play_df)
SLOT_SIZE = TOTAL // TOTAL_SLOTS
IDX_START = MY_SLOT * SLOT_SIZE
IDX_END   = IDX_START + SLOT_SIZE if MY_SLOT < TOTAL_SLOTS - 1 else TOTAL

play_df = play_df.iloc[IDX_START:IDX_END].reset_index(drop=True)
print(f'\n슬롯 {MY_SLOT} 담당: {IDX_START:,} ~ {IDX_END:,}  ({len(play_df):,}개)')
play_df.head()

기존 샘플 로드: 93,466개

슬롯 0 담당: 0 ~ 18,693  (18,693개)


,play_id,pitcher_id,game_pk,season
0,a3381ae9-0f84-49b3-8781-7c71f8cce66d,425794,632314,2021
1,d5a3763f-71e9-4872-8d3d-cfc8ab8d4046,425794,632314,2021
2,3991c163-3bbe-4851-91d1-5637b01b9480,425794,632314,2021
3,8c470367-fa84-4ba5-bc15-5f9710c17bd2,425794,632314,2021
4,fd462b59-2160-426a-8fb5-05fadcd25459,425794,632314,2021


## Step 2. MP4 다운로드 → zip 저장

In [5]:
def get_cdn_url(play_id, retry=3):
    url = f'https://baseballsavant.mlb.com/sporty-videos?playId={play_id}'
    for attempt in range(retry):
        try:
            res  = requests.get(url, headers=REQ_HEADERS, timeout=15)
            soup = BeautifulSoup(res.text, 'html.parser')
            video = soup.find('video')
            if video and video.find('source'):
                return video.find('source').get('src')
            return None
        except Exception:
            time.sleep(2 ** attempt)
    return None

def download_mp4(cdn_url, save_path, retry=3):
    for attempt in range(retry):
        try:
            with requests.get(cdn_url, headers=REQ_HEADERS, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(save_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
            return True
        except Exception:
            if os.path.exists(save_path):
                os.remove(save_path)
            time.sleep(2 ** attempt)
    return False

print('다운로드 함수 정의 완료')

다운로드 함수 정의 완료


In [6]:
# 체크포인트 로드
if os.path.exists(PROGRESS_CSV):
    progress = pd.read_csv(PROGRESS_CSV)
    done_ids = set(progress[progress['status'] == 'done']['play_id'].tolist())
    print(f'체크포인트 로드: 완료 {len(done_ids):,}개')
else:
    progress = pd.DataFrame(columns=['play_id', 'pitcher_id', 'season', 'batch_id', 'status'])
    done_ids = set()
    print('체크포인트 없음 → 처음부터 시작')

remaining = play_df[~play_df['play_id'].isin(done_ids)].reset_index(drop=True)

existing_zips = list(Path(VIDEO_DIR).glob(f'batch_slot{MY_SLOT}_*.zip'))
start_batch   = len(existing_zips)

print(f'전체 play_id : {len(play_df):,}개')
print(f'이미 완료    : {len(done_ids):,}개')
print(f'남은 작업    : {len(remaining):,}개')
print(f'기존 zip     : {start_batch}개')
print(f'예상 시간    : 약 {len(remaining) * 4 / 3600:.1f}시간')

체크포인트 로드: 완료 21,969개
전체 play_id : 18,693개
이미 완료    : 21,969개
남은 작업    : 4,085개
기존 zip     : 72개
예상 시간    : 약 4.5시간


In [ ]:
new_records = []

for batch_idx, batch_start in enumerate(range(0, len(remaining), BATCH_SIZE)):
    batch_num  = start_batch + batch_idx + 1
    batch_name = f'batch_slot{MY_SLOT}_{batch_num:04d}'
    zip_path   = os.path.join(VIDEO_DIR, f'{batch_name}.zip')

    if os.path.exists(zip_path):
        print(f'[{batch_name}] 이미 존재 → 건너뜀')
        continue

    batch_df  = remaining.iloc[batch_start:batch_start + BATCH_SIZE]
    batch_dir = os.path.join(TEMP_DIR, batch_name)
    os.makedirs(batch_dir, exist_ok=True)

    print(f'\n[{batch_name}] {len(batch_df)}개 시작...')
    batch_done = batch_fail = 0

    for i, (_, row) in enumerate(batch_df.iterrows()):
        play_id    = row['play_id']
        pitcher_id = row.get('pitcher_id', '')
        season     = row['season']
        mp4_path   = os.path.join(batch_dir, f'{play_id}.mp4')

        status = 'fail'
        try:
            cdn_url = get_cdn_url(play_id)
            if cdn_url and download_mp4(cdn_url, mp4_path):
                status = 'done'
                batch_done += 1
            else:
                batch_fail += 1
        except Exception:
            batch_fail += 1

        new_records.append({
            'play_id': play_id, 'pitcher_id': pitcher_id,
            'season': season, 'batch_id': batch_name, 'status': status
        })

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(batch_df)} | 성공 {batch_done} / 실패 {batch_fail}')

        time.sleep(1.0)

    # zip 압축
    print(f'[{batch_name}] zip 압축 중...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for mp4 in Path(batch_dir).glob('*.mp4'):
            zf.write(mp4, mp4.name)

    zip_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'[{batch_name}] 완료 {zip_mb:.1f} MB | 성공 {batch_done} / 실패 {batch_fail}')

    shutil.rmtree(batch_dir)

    # 체크포인트 저장
    progress = pd.concat([progress, pd.DataFrame(new_records)], ignore_index=True)
    progress.to_csv(PROGRESS_CSV, index=False)
    new_records = []

print('\n전체 완료!')


[batch_slot0_0073] 200개 시작...
  20/200 | 성공 4 / 실패 16
  40/200 | 성공 18 / 실패 22
  60/200 | 성공 38 / 실패 22
  80/200 | 성공 58 / 실패 22
  100/200 | 성공 78 / 실패 22
  120/200 | 성공 98 / 실패 22
  140/200 | 성공 118 / 실패 22
  160/200 | 성공 138 / 실패 22
  180/200 | 성공 158 / 실패 22
  200/200 | 성공 178 / 실패 22
[batch_slot0_0073] zip 압축 중...
[batch_slot0_0073] 완료 981.0 MB | 성공 178 / 실패 22
[batch_slot0_0074] 이미 존재 → 건너뜀

[batch_slot0_0075] 200개 시작...
  20/200 | 성공 20 / 실패 0
  40/200 | 성공 40 / 실패 0
  60/200 | 성공 60 / 실패 0
  80/200 | 성공 80 / 실패 0
  100/200 | 성공 100 / 실패 0
  120/200 | 성공 120 / 실패 0
  140/200 | 성공 140 / 실패 0
  160/200 | 성공 160 / 실패 0
  180/200 | 성공 180 / 실패 0
  200/200 | 성공 200 / 실패 0
[batch_slot0_0075] zip 압축 중...


## Step 3. 현황 확인

In [ ]:
progress = pd.read_csv(PROGRESS_CSV)
print(f'=== 슬롯 {MY_SLOT} ({MY_SEASON}시즌) 다운로드 현황 ===')
print(progress['status'].value_counts())

if 'pitcher_id' in progress.columns:
    n_pitchers = progress[progress['status'] == 'done']['pitcher_id'].nunique()
    print(f'\n투수 수: {n_pitchers}명')

zips = sorted(Path(VIDEO_DIR).glob(f'batch_slot{MY_SLOT}_*.zip'))
total_gb = sum(z.stat().st_size for z in zips) / 1024**3
print(f'\nzip 파일: {len(zips)}개 / 총 {total_gb:.2f} GB')

---
## 부록 A. play_id → game_pk 매핑 (대체 방법, 구 `build_play_game_map.py`)

> 위 크롤링은 game_pk를 함께 확보하지만, **play_id만 있고 game_pk가 없을 때**(초기 수집분 등)를 위한 백업 매핑 도구.
> MLB statsapi `game/{game_pk}/content` API를 **병렬(10 workers)** 호출해 play_id↔game_pk를 역매핑 → `play_game_map.csv`.
> 아래 셀은 필요할 때만 실행 (평소 수집 흐름엔 불필요).

In [ ]:
# [부록 A] play_id → game_pk 매핑 생성 (필요 시 실행)
import csv, re, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

# 경로: 06의 DATA_DIR / PARQUET_DIR 재사용 (팀원 PC 하드코딩 제거)
PLAY_ID_FILES = [os.path.join(DATA_DIR, 'play_ids_all.csv')]   # play_id, pitcher_id, season 포함
MAP_OUTPUT    = os.path.join(DATA_DIR, 'play_game_map.csv')
UUID_PAT = re.compile(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}')
HEADERS  = {'User-Agent': 'Mozilla/5.0'}

def build_play_game_map(statcast_parquets, workers=10, save_every=500):
    # 1. play_id → (pitcher_id, season)
    play_meta = {}
    for path in PLAY_ID_FILES:
        with open(path) as f:
            for row in csv.DictReader(f):
                play_meta[row['play_id']] = (row['pitcher_id'], row['season'])
    all_play_ids = set(play_meta)
    print(f'play_id: {len(all_play_ids):,}개')

    # 2. statcast에서 game_pk 목록
    import pyarrow.parquet as pq
    game_pks = set()
    for pf in statcast_parquets:
        for gk in pq.read_table(pf, columns=['game_pk'])['game_pk'].to_pylist():
            if gk is not None:
                game_pks.add(int(gk))
    print(f'game_pk: {len(game_pks):,}개')

    # 3. game content API 병렬 호출 → play_id 매칭
    def fetch(game_pk):
        try:
            req = urllib.request.Request(
                f'https://statsapi.mlb.com/api/v1/game/{game_pk}/content', headers=HEADERS)
            text = urllib.request.urlopen(req, timeout=10).read().decode('utf-8', 'ignore')
            return game_pk, set(UUID_PAT.findall(text)) & all_play_ids
        except Exception:
            return game_pk, set()

    results = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(fetch, gk): gk for gk in sorted(game_pks)}
        for n, fut in enumerate(as_completed(futures), 1):
            game_pk, matched = fut.result()
            for pid in matched:
                pitcher_id, season = play_meta[pid]
                results.append({'play_id': pid, 'game_pk': game_pk,
                                'pitcher_id': pitcher_id, 'season': season})
            if n % save_every == 0 or n == len(game_pks):
                pd.DataFrame(results).to_csv(MAP_OUTPUT, index=False)
                print(f'  {n:,}/{len(game_pks):,} | 매핑 {len(results):,}개')
    print(f'완료: {MAP_OUTPUT} ({len(results):,}개)')
    return results

# 실행 예 (필요 시 주석 해제):
# parquets = sorted(glob.glob(os.path.join(PARQUET_DIR, 'statcast_*.parquet')))
# build_play_game_map(parquets)
print('부록 A 함수 정의 완료 (build_play_game_map)')

---
## 부록 B. 크롤링 버그 검증 — `group_by=name` 진단 (구 `verify_crawl_fix.py`)

> **문제**: 경기당 매핑되는 영상이 2~10개로 들쭉날쭉(원래 ~15구여야 함).
> **가설**: 크롤링 URL의 `group_by=name`이 초구 play_id를 깎는다.
> **검증**: 같은 (pitcher, game_pk)로 `group_by` **유/무를 A/B 비교**해 play_id 개수 차이를 센다.
> → group_by 제거 시 더 많이 나오면 그게 범인. (데이터 수집 품질을 코드로 진단한 사례)

In [ ]:
# [부록 B] group_by=name 이 초구 play_id를 깎는지 A/B 검증 (필요 시 실행)
def build_verify_url(pitcher, game_pk, season, group_by):
    """group_by 파라미터만 다르게 해서 URL 생성"""
    return (
        f'https://baseballsavant.mlb.com/statcast_search'
        f'?hfPT=&hfAB=&hfGT=R%7C&hfPR=&hfZ=&hfStadium=&hfBBL=&hfNewZones='
        f'&hfPull=&hfC=&hfSea={season}%7C&hfSit=&player_type=pitcher'
        f'&hfOuts=&home_road=&pitcher_throws=&batter_stands=&hfSA='
        f'&hfEventOuts=&hfEventRuns=&game_date_gt=&game_date_lt=&hfMo='
        f'&hfTeam=&hfOpponent=&hfRO=&position=&hfInfield=&hfOutfield='
        f'&hfInn=&hfBBT=&hfFlag='
        f'&pitchers_lookup%5B%5D={pitcher}&game_pk={game_pk}'
        f'&metric_1=&group_by={group_by}&min_pitches=0&min_results=0&min_pas=0'
        f'&sort_col=pitches&player_event_sort=api_p_release_speed&sort_order=desc'
        f'&type=details&player_id={pitcher}'
    )

def count_play_ids(url):
    res = requests.get(url, headers=REQ_HEADERS, timeout=30)
    ids = set()
    for tag in BeautifulSoup(res.text, 'html.parser').find_all('a', href=True):
        m = re.search(r'playId=([0-9a-f\-]{36})', tag['href'])
        if m:
            ids.add(m.group(1))
    return ids

def verify_group_by_bug(pitcher_id, game_pk, season):
    print(f'테스트: pitcher={pitcher_id}, game_pk={game_pk}, season={season}\n')
    grouped = count_play_ids(build_verify_url(pitcher_id, game_pk, season, 'name'))
    nogroup = count_play_ids(build_verify_url(pitcher_id, game_pk, season, ''))
    print(f'[group_by=name  ] play_id 수: {len(grouped)}')
    print(f'[group_by=(없음)] play_id 수: {len(nogroup)}')
    if len(nogroup) > len(grouped):
        print(f'\n✅ group_by 제거 시 {len(nogroup)-len(grouped)}개 더 확보 → group_by=name 이 범인')
    elif len(nogroup) == len(grouped):
        print('\n⚠️ 차이 없음 → group_by가 원인 아님')
    else:
        print('\n❓ 예상과 반대 → 수동 확인 필요')

# 실행 예 (play_game_map.csv 등에서 실제 값 골라 넣기):
# verify_group_by_bug(pitcher_id=645261, game_pk=716367, season=2023)
print('부록 B 함수 정의 완료 (verify_group_by_bug)')